# NB-12 — OGC Datacube Dimensions End-to-End

**Goal:** Register the standard reusable dimensions via the `common_dimensions` preset,
then explore and query them through the OGC Dimensions API.

**What this notebook covers:**

| Section | Description |
|---------|-------------|
| 1. Setup | Environment, client, required env vars |
| 2. Apply preset | `POST /configs/presets/common_dimensions` — registers dimensions + triggers materialisation |
| 3. Discover dimensions | `GET /dimensions` — list all registered dimensions |
| 4. Dekadal pagination | `GET /dimensions/{id}/items` — page through 36 periods |
| 5. Range search | `GET /dimensions/{id}/search?min=&max=` — filter by code range |
| 6. Inverse lookup | `GET /dimensions/{id}/search?exact=` — map a value back to a member |
| 7. Admin hierarchy | `GET /dimensions/{id}/children` + `/ancestors` — continent→country |
| 8. Monitor task | Poll `GET /processes/jobs/{job_id}` until `successful` |
| 9. Verify results | Confirm member counts after materialisation |

**Prerequisites:**
- Platform running with `extension_dimensions` and `task_dimensions_materialize` in SCOPE
- `configs` extension loaded (provides `/configs/presets/…`)

**Env vars (set in `.env` or shell):**

| Variable | Purpose | Default |
|----------|---------|--------|
| `DYNASTORE_BASE_URL` | Platform base URL | `http://localhost:8080` |
| `DYNASTORE_TOKEN` | Bearer token (admin) | auto-provisioned via IDP if not set |
| `IDP_PUBLIC_URL` | Token issuer base URL | — |
| `IDP_CLIENT_ID` | Client ID for `client_credentials` | — |
| `IDP_CLIENT_SECRET` | Client secret | — |

## 1. Setup

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = os.environ.get("DYNASTORE_TOKEN")
HEADERS = {"Accept": "application/json"}
if TOKEN:
    HEADERS["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=HEADERS, timeout=30)

DIM_BASE = "/dimensions"  # router prefix from DimensionsExtension (dimensions_extension.py:787)

print(f"Base URL : {BASE_URL}")
print(f"Auth     : {'token present' if TOKEN else 'no token (anonymous)'}")

## 2. Apply the `common_dimensions` preset

The `common_dimensions` preset (registered in `extensions/dimensions/presets/__init__.py`)
does two things in one call:

1. **Skeleton phase (sync):** Creates RECORDS collections for each registered dimension
   in the shared `_dimensions_` catalog. Clients can discover dimensions immediately.
2. **Fill phase (async):** Enqueues the `dimensions_materialize` OGC Process job to
   populate member records. The preset returns a `job_id` in the applied descriptor
   so you can poll the job without a second API call.

Route: `POST /configs/presets/{preset_name}` (platform tier)
Source: `packages/extensions/configs/src/dynastore/extensions/configs/presets_api.py:440`

In [ ]:
# Apply the common_dimensions preset at platform tier.
# POST /configs/presets/common_dimensions  (presets_api.py:440)
resp = client.post("/configs/presets/common_dimensions")
print(f"Status: {resp.status_code}")
assert resp.status_code in (200, 201, 409), (
    f"Unexpected status applying preset: {resp.status_code} — {resp.text[:400]}"
)

body = resp.json()
print(json.dumps(body, indent=2)[:800])

# Extract job_id for the materialisation task (may be nested under 'descriptor' or 'applied')
_descriptor = body.get("descriptor") or body.get("applied") or body
_task_info = _descriptor.get("task") or _descriptor.get("job") or {}
job_id = _task_info.get("jobID") or _task_info.get("job_id")
print(f"\nMaterialisation job_id: {job_id or '(not returned — check body above)'}")

## 3. Discover dimensions

`GET /dimensions` returns all registered dimensions as a FeatureCollection of collections.
Each entry carries a `provider` object describing its type and config.

Route: `GET /dimensions`
Source: `packages/extensions/dimensions/src/dynastore/extensions/dimensions/dimensions_extension.py:787`
(the `dimensions_router` from `ogc_dimensions.api.routes` is included at prefix `/dimensions`)

In [ ]:
# GET /dimensions  — list all registered dimensions
resp = client.get(DIM_BASE)
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

dims_payload = resp.json()
collections = dims_payload.get("collections", [])

print(f"\nRegistered dimensions ({len(collections)}):")
for d in collections:
    prov = d.get("provider", {})
    ptype = prov.get("type", "-")
    hierarchical = prov.get("hierarchical", False)
    invertible = prov.get("invertible", False)
    cfg = prov.get("config", {})
    print(
        f"  id={d.get('id')!r:30}  provider={ptype!r:15}"
        f"  hierarchical={hierarchical}  invertible={invertible}"
        + (f"  period_days={cfg.get('period_days')}" if cfg.get('period_days') else "")
    )

# Locate the dekadal temporal and admin dimensions for later steps
dekadal_dim = next(
    (d for d in collections
     if d.get("provider", {}).get("config", {}).get("period_days") == 10),
    None,
)
admin_dim = next(
    (d for d in collections
     if d.get("provider", {}).get("hierarchical")),
    None,
)

DEKADAL_ID = dekadal_dim["id"] if dekadal_dim else "temporal-dekadal"
ADMIN_ID = admin_dim["id"] if admin_dim else "world-admin"

print(f"\nUsing dekadal dimension : {DEKADAL_ID!r}")
print(f"Using admin dimension   : {ADMIN_ID!r}")

## 4. Dekadal pagination

The dekadal temporal dimension has 36 periods per year (3 per month: D1 days 1-10, D2 days
11-20, D3 days 21-end). For multi-year datasets the total can exceed 1 000 members, so
the API paginates via `links[rel=next]`.

Route: `GET /dimensions/{dimension_id}/items?limit=&offset=`
Source: `packages/extensions/dimensions/src/dynastore/extensions/dimensions/dimensions_extension.py:787`
(included via `dimensions_router` from `ogc_dimensions.api.routes`)

In [ ]:
# First page of items — limit=10
# GET /dimensions/{dimension_id}/items
resp = client.get(f"{DIM_BASE}/{DEKADAL_ID}/items", params={"limit": 10, "offset": 0})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
page_features = data.get("features", [])
number_matched = data.get("numberMatched")

print(f"numberMatched : {number_matched}")
print(f"Page size     : {len(page_features)}")
print()
for f in page_features[:5]:
    props = f["properties"]
    interval = props.get("time", {}).get("interval", [None, None])
    print(f"  {f['id']:12}  {interval[0]} — {interval[1]}")

# Show links
links = data.get("links", [])
next_link = next((lnk["href"] for lnk in links if lnk.get("rel") == "next"), None)
print(f"\nrel=next present: {next_link is not None}")

In [ ]:
# Follow rel=next until all pages are exhausted
# Demonstrates the OGC pagination pattern (dimension-pagination conformance)
all_periods = list(page_features)
url = f"{DIM_BASE}/{DEKADAL_ID}/items"
params = {"limit": 12, "offset": 0}
page_num = 0

while True:
    resp = client.get(url, params=params)
    assert resp.status_code == 200, f"Page {page_num}: {resp.status_code}: {resp.text}"
    data = resp.json()
    batch = data.get("features", [])
    # avoid double-counting the first page already fetched above
    if page_num == 0:
        all_periods = batch  # reset from above
    else:
        all_periods.extend(batch)
    page_num += 1
    print(f"  Page {page_num}: {len(batch)} items (running total: {len(all_periods)})")

    next_lnk = next(
        (lnk["href"] for lnk in data.get("links", []) if lnk.get("rel") == "next"),
        None,
    )
    if not next_lnk:
        break
    # next_lnk is an absolute URL; extract path+query for the httpx client
    from urllib.parse import urlparse, parse_qs
    parsed = urlparse(next_lnk)
    url = parsed.path
    params = {k: v[0] for k, v in parse_qs(parsed.query).items()}

print(f"\nTotal periods retrieved: {len(all_periods)}")

## 5. Range search

The `search` endpoint lets you filter dimension members by code range (`min`/`max`).
For the dekadal dimension, period codes follow the pattern `YYYY-KNN` (e.g. `2024-K01`).

Route: `GET /dimensions/{dimension_id}/search?min=&max=`
Source: `packages/extensions/dimensions/src/dynastore/extensions/dimensions/dimensions_extension.py:793`
(geoid-owned `/search` route registered before the upstream router)

In [ ]:
# All 36 dekadal periods of 2024
# GET /dimensions/{dimension_id}/search?min=2024-K01&max=2024-K36
resp = client.get(
    f"{DIM_BASE}/{DEKADAL_ID}/search",
    params={"min": "2024-K01", "max": "2024-K36"},
)
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
periods_2024 = data.get("features", [])

print(f"2024 dekadal periods: {len(periods_2024)}")
assert len(periods_2024) == 36, f"Expected 36, got {len(periods_2024)}"

for p in periods_2024[:6]:
    props = p["properties"]
    interval = props.get("time", {}).get("interval", [None, None])
    print(f"  {p['id']:12}  {interval[0]} — {interval[1]}")
print(f"  ... ({len(periods_2024) - 6} more)")

# February edge case: 2024 is a leap year; K06 covers Feb 21–29 (9 days)
resp_feb = client.get(
    f"{DIM_BASE}/{DEKADAL_ID}/search",
    params={"min": "2024-K04", "max": "2024-K06"},
)
assert resp_feb.status_code == 200
feb = resp_feb.json().get("features", [])
assert len(feb) == 3, f"Expected 3 February dekads, got {len(feb)}"
k06 = next(p for p in feb if p["id"] == "2024-K06")
k06_end = k06["properties"].get("time", {}).get("interval", [None, None])[1]
assert k06_end == "2024-02-29", f"Expected leap-year end 2024-02-29, got {k06_end}"
print(f"\nLeap-year edge case: K06 end = {k06_end} — correct")

# Out-of-range year: empty result, not 404
resp_oor = client.get(
    f"{DIM_BASE}/{DEKADAL_ID}/search",
    params={"min": "2099-K01", "max": "2099-K36"},
)
assert resp_oor.status_code == 200
assert len(resp_oor.json().get("features", [])) == 0
print("Out-of-range year returns 200 + empty result — correct")

## 6. Inverse lookup

Given an arbitrary date, find the dekad that contains it.
Uses `search?exact=` with the dimension's invertible protocol.

Route: `GET /dimensions/{dimension_id}/search?exact=`
Source: `packages/extensions/dimensions/src/dynastore/extensions/dimensions/dimensions_extension.py:793`

In [ ]:
# Inverse lookup: which dekad contains 2024-01-15?
# GET /dimensions/{dimension_id}/search?exact=2024-01-15
test_dates = ["2024-01-15", "2024-02-28", "2024-06-21", "2024-12-31"]

print("Inverse lookups (date → dekad code):")
for date in test_dates:
    resp = client.get(
        f"{DIM_BASE}/{DEKADAL_ID}/search",
        params={"exact": date},
    )
    assert resp.status_code == 200, f"{date}: {resp.status_code}: {resp.text}"
    features = resp.json().get("features", [])
    if features:
        f = features[0]
        interval = f["properties"].get("time", {}).get("interval", [None, None])
        print(f"  {date} → {f['id']:12}  range: {interval[0]} — {interval[1]}")
    else:
        print(f"  {date} → (no member — dimension may not cover this date)")

## 7. Admin hierarchy navigation

Hierarchical dimensions (e.g. `world-admin`: continent → country) expose
two additional endpoints:

- `GET /dimensions/{id}/children?parent=CODE` — direct children of a node
- `GET /dimensions/{id}/ancestors?member=CODE` — ancestor chain (breadcrumb)

Both return GeoJSON FeatureCollections. Unknown codes return 200 + empty result.

Routes: `GET /dimensions/{dimension_id}/children` and `/ancestors`
Source: `packages/extensions/dimensions/src/dynastore/extensions/dimensions/dimensions_extension.py:787`
(included via `dimensions_router` from `ogc_dimensions.api.routes`)

In [ ]:
# Inspect the admin dimension
resp = client.get(DIM_BASE)
assert resp.status_code == 200
collections = resp.json().get("collections", [])
admin_meta = next((d for d in collections if d["id"] == ADMIN_ID), None)

if admin_meta is None:
    print(f"Admin dimension '{ADMIN_ID}' not found — check DIM_BASE listing above.")
else:
    prov = admin_meta.get("provider", {})
    print(f"id          : {admin_meta.get('id')}")
    print(f"title       : {admin_meta.get('title')}")
    print(f"provider    : {prov.get('type')}")
    print(f"hierarchical: {prov.get('hierarchical')}")
    assert prov.get("hierarchical"), "Expected a hierarchical dimension"

In [ ]:
# Children of Africa (AFR)
# GET /dimensions/{dimension_id}/children?parent=AFR
resp = client.get(f"{DIM_BASE}/{ADMIN_ID}/children", params={"parent": "AFR"})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
afr_children = data.get("features", [])
country_ids = [n["id"] for n in afr_children]

print(f"Africa child nodes ({len(afr_children)}):")
for node in afr_children[:8]:
    props = node["properties"]
    print(f"  id={node['id']:6}  label={props.get('title', '—')}")

assert len(afr_children) > 0, "Africa (AFR) must have at least one child node"

# Leaf node: country-level nodes should have no further children
leaf = country_ids[0]
resp_leaf = client.get(f"{DIM_BASE}/{ADMIN_ID}/children", params={"parent": leaf})
assert resp_leaf.status_code == 200
leaf_children = resp_leaf.json().get("features", [])
print(f"\nChildren of leaf node '{leaf}': {len(leaf_children)} (expected 0)")

# Unknown parent: 200 + empty
resp_unk = client.get(f"{DIM_BASE}/{ADMIN_ID}/children", params={"parent": "DOES_NOT_EXIST"})
assert resp_unk.status_code == 200
assert len(resp_unk.json().get("features", [])) == 0
print("Unknown parent returns 200 + empty list — correct")

In [ ]:
# Ancestor chain (breadcrumb) for Ethiopia
# GET /dimensions/{dimension_id}/ancestors?member=ETH
target = "ETH"
resp = client.get(f"{DIM_BASE}/{ADMIN_ID}/ancestors", params={"member": target})
print(resp.status_code)
assert resp.status_code == 200, f"Expected 200, got {resp.status_code}: {resp.text}"

data = resp.json()
ancestors = data.get("features", [])
ancestor_ids = [n["id"] for n in ancestors]

print(f"Ancestor chain for {target}: {ancestor_ids}")
assert "AFR" in ancestor_ids, f"Expected AFR in ancestor chain of {target}, got {ancestor_ids}"
print(f"Breadcrumb: {' > '.join(ancestor_ids)} > {target}")

## 8. Monitor the materialisation task

The `common_dimensions` preset enqueues a `dimensions_materialize` OGC Process job.
Poll `GET /processes/jobs/{job_id}` until `status` reaches a terminal state.

Route: `GET /processes/jobs/{job_id}`
Terminal states: `successful`, `failed`, `dismissed`

In [ ]:
TERMINAL = {"successful", "failed", "dismissed"}

if not job_id:
    print("No job_id available from preset apply — skipping poll.")
    print("Trigger manually via: POST /processes/dimensions_materialize/execution")
else:
    print(f"Polling job {job_id} …")
    _max_polls = 30
    for _i in range(_max_polls):
        r = client.get(f"/processes/jobs/{job_id}")
        r.raise_for_status()
        s = r.json()
        status = s.get("status")
        progress = s.get("progress")
        msg = s.get("message", "")
        print(f"  [{_i+1:2d}] status={status}  progress={progress}  {msg[:80]}")
        if status in TERMINAL:
            break
        time.sleep(3)

    print(f"\nFinal status: {status}")
    if status == "failed":
        r2 = client.get(f"/processes/jobs/{job_id}/results")
        print(r2.text[:600])

## 9. Verify member counts

After materialisation, each dimension collection in `_dimensions_` should have
its member items populated. Verify via `GET /records/catalogs/_dimensions_/collections/{dim_id}/items`.

Route: `GET /records/catalogs/{catalog_id}/collections/{collection_id}/items`
(OGC API Records items endpoint for the shared `_dimensions_` catalog)

In [ ]:
DIMENSIONS_CATALOG = "_dimensions_"

# List all dimension collections
resp = client.get("/collections", params={"catalog": DIMENSIONS_CATALOG})
if resp.status_code != 200:
    print(f"Collection listing returned {resp.status_code} — materialisation may still be running.")
else:
    dim_collections = resp.json().get("collections", [])
    print(f"Dimension collections in {DIMENSIONS_CATALOG!r}: {len(dim_collections)}")
    print()

    for c in dim_collections:
        cid = c.get("id")
        it = client.get(
            f"/records/catalogs/{DIMENSIONS_CATALOG}/collections/{cid}/items",
            params={"limit": 1},
        )
        if it.is_success:
            total = it.json().get("numberMatched", "?")
        else:
            total = f"HTTP {it.status_code}"
        print(f"  {cid:30}  numberMatched={total}")

## Cleanup

To roll back the `common_dimensions` preset:

```
DELETE /configs/presets/common_dimensions
```

This removes only the dimension collection skeletons registered by the preset.
Collections with materialised members are protected (skipped, not deleted).
Shared `_dimensions_` catalog is never deleted (managed_catalog=False).

In [ ]:
client.close()
print("Client closed.")